## Homework

In this dataset our desired target for classification task will be converted variable - has the client signed up to the platform or not.

## 1. Data Preparation

Check if the missing values are presented in the features.
If there are missing values:
- For categorical features, replace them with 'NA'
- For numerical features, replace with with 0.0

In [120]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

In [121]:
data = 'wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

In [122]:
!wget -nc $data

Prepended http:// to 'wget'
--2026-09-28 18:28:23--  http://wget/
Resolving wget (wget)... failed: Name or service not known.
wget: unable to resolve host address ‘wget’
File ‘course_lead_scoring_2026.csv’ already there; not retrieving.



In [123]:
data = pd.read_csv('course_lead_scoring_2026.csv')

In [124]:
data.columns = data.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(data.select_dtypes(include=['object', 'string']).columns)

for c in categorical_columns:
    data[c] = data[c].str.lower().str.replace(' ', '_')

In [125]:
data.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [126]:
len(data)

5000

In [127]:
data.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [128]:
data.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [129]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']

numerical = ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']

- For categorical features, replace them with 'NA'
- For numerical features, replace with with 0.0

In [130]:
data[categorical] = data[categorical].fillna('NA')
data[categorical].isnull().sum()

lead_source          0
industry             0
employment_status    0
location             0
dtype: int64

In [131]:
data[numerical] = data[numerical].fillna(0)
data[numerical].isnull().sum()

number_of_courses_viewed    0
annual_income               0
interaction_count           0
lead_score                  0
dtype: int64

## Question 1
What is the most frequent observation (mode) for the column industry?

- NA
- technology
- healthcare
- retail

Answer
- technology

In [132]:
data['industry'].mode()

0    technology
Name: industry, dtype: str

## Question 2
Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- interaction_count and lead_score
- number_of_courses_viewed and lead_score
- number_of_courses_viewed and interaction_count
- annual_income and interaction_count

Only consider the pairs above when answering this question.

Answer
- interaction_count and lead_score

In [133]:
data[numerical].corr().abs()

,number_of_courses_viewed,annual_income,interaction_count,lead_score
number_of_courses_viewed,1.000000,0.161300,0.721609,0.757204
annual_income,0.161300,1.000000,0.122842,0.229496
interaction_count,0.721609,0.122842,1.000000,0.915746
lead_score,0.757204,0.229496,0.915746,1.000000


## Split the data
Split your data in train/val/test sets with 60%/20%/20% distribution.
Use Scikit-Learn for that (the train_test_split function) and set the seed to 42.
Make sure that the target value converted is not in your dataframe

In [134]:
from sklearn.model_selection import train_test_split

In [135]:
df_full_train, df_test = train_test_split(data, test_size = 0.2, random_state=42)
len(df_full_train), len(df_test)

(4000, 1000)

In [136]:
df_train, df_val = train_test_split(df_full_train, test_size = 0.25, random_state=42)
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [137]:
df_train.reset_index(drop=True)
df_val.reset_index(drop=True)
df_test.reset_index(drop=True)

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,social_media,retail,employed,south_america,41247.0,3,7,0.61,1
1,organic_search,healthcare,employed,north_america,70825.0,3,6,0.62,0
2,social_media,healthcare,employed,south_america,58488.0,3,4,0.56,1
3,NA,education,NA,europe,30138.0,1,4,0.48,0
4,NA,retail,employed,asia,53869.0,2,3,0.49,1
...,...,...,...,...,...,...,...,...,...
995,organic_search,manufacturing,employed,NA,65099.0,1,4,0.53,1
996,social_media,healthcare,student,europe,38954.0,0,0,0.15,0
997,organic_search,education,self_employed,asia,48768.0,2,5,0.47,1
998,social_media,NA,student,asia,0.0,2,2,0.35,1


In [138]:
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

In [139]:
del(df_train['converted'])
del(df_val['converted'])
del(df_test['converted'])

## Question 3
- Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using round(score, 2).

Which of these variables has the biggest mutual information score?

- industry
- location
- lead_source
- employment_status

Answer
- lead_source

In [140]:
from sklearn.metrics import mutual_info_score

In [141]:
for c in categorical:
    mi = mutual_info_score(df_full_train['converted'], df_full_train[c])
    mi = round(mi,2)
    print(f"{c}:{mi:}")

lead_source:0.03
industry:0.0
employment_status:0.02
location:0.0



## Question 4
- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
    - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
    - model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

Calculate the accuracy on the validation dataset and round it to 2 decimal digits.


What accuracy did you get?
- 0.64
- 0.74
- 0.84
- 0.94

Answer
- 0.74

In [142]:
from sklearn.feature_extraction import DictVectorizer

In [143]:
dv = DictVectorizer(sparse=False)

In [144]:
dict_train = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(dict_train)

In [145]:
from sklearn.linear_model import LogisticRegression

In [146]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

In [147]:
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The 

In [148]:
dict_val = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(dict_val)

In [149]:
y_pred = model.predict_proba(X_val)[:,1]

In [150]:
converts = y_pred >= 0.5
round((converts == y_val).mean(),2)

np.float64(0.64)

## Question 5
- Let's find the least useful feature using the feature elimination technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

- 'lead_source'
- 'number_of_sources_viewed'
- 'interaction_count'

Note: The difference doesn't have to be positive.

Answer
- 'number_of_courses_viewed'

In [151]:
converts = y_pred >= 0.5
original_accuracy = round((converts == y_val).mean(),4)
original_accuracy

np.float64(0.645)

In [152]:
all_features = categorical + numerical

In [153]:
cols = list(set(categorical) - {'lead_source'})
dict_train = df_train[cols + numerical].to_dict(orient='records')
X_train = dv.fit_transform(dict_train)

In [154]:
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The 

In [155]:
cols = list(set(categorical) - {'lead_source'})
dict_val = df_val[cols + numerical].to_dict(orient='records')
X_val = dv.transform(dict_val)

In [156]:
y_pred_lead = model.predict_proba(X_val)[:,1]
converts = y_pred_lead >= 0.5
round((converts == y_val).mean(),2)

np.float64(0.64)

In [157]:
def evaluate_without_feature(feature_to_remove, df_train,df_val,y_train,y_val,categorical, numerical):
    
    # 1. Create feature list without the target feature
    features = [col for col in (categorical + numerical) if col != feature_to_remove]
    
    # 2. Vectorize train data
    dict_train = df_train[features].to_dict(orient='records')
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dict_train)
    
    # 3. Fit Model
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    # 4. Vectorize validation data using the SAME features
    dict_val = df_val[features].to_dict(orient='records')
    X_val = dv.transform(dict_val)
    
    # 5. Predict & evaluate
    y_pred = model.predict_proba(X_val)[:,1]
    converts = y_pred >= 0.5
    accuracy = (converts == y_val).mean()

    return round(accuracy,4)

In [158]:
lead_accuracy = evaluate_without_feature('lead_source',df_train=df_train,df_val=df_val,y_train=y_train,y_val=y_val,categorical=categorical,numerical=numerical)
lead_diff = original_accuracy - lead_accuracy
lead_accuracy, lead_diff

(np.float64(0.642), np.float64(0.0030000000000000027))

In [159]:
diffs = {}

# Compute differences for each categorical feature
for feature in all_features:
    acc = evaluate_without_feature(feature, df_train=df_train, df_val=df_val, y_train=y_train, y_val=y_val, categorical=categorical, numerical=numerical)
    diffs[feature] = original_accuracy - acc

# Sort features by highest impact (largest drop in accuracy) to lowest
ranked_diffs = dict(sorted(diffs.items(), key=lambda item: item[1], reverse=True))

for feature, diff in ranked_diffs.items():
    print(f"{feature}: {diff:+.4f}")

interaction_count: +0.0440
lead_source: +0.0030
number_of_courses_viewed: +0.0020
lead_score: +0.0010
industry: +0.0000
employment_status: +0.0000
location: +0.0000
annual_income: -0.0790


## Question 6
- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter C: [0.01, 0.1, 1, 10, 100].
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these C leads to the best accuracy on the validation set?

- 0.01
- 0.1
- 1
- 10
- 100

Note: If there are multiple options, select the smallest C.

Answer
- 0.01

In [160]:
from sklearn.linear_model import LogisticRegressionCV

In [161]:
model = LogisticRegressionCV(solver='liblinear', Cs= [0.01, 0.1, 1, 10, 100], max_iter=1000, random_state=42)

In [162]:
dict_train = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(dict_train)

In [163]:
model.fit(X_train, y_train)

/home/pc/.local/share/virtualenvs/zindi-9FaJJdpi/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:2092: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/pc/.local/share/virtualenvs/zindi-9FaJJdpi/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:2137: FutureWarning: The default value of the parameter 'scoring' will change from None, i.e. accuracy, to 'neg_log_loss' in version 1.11. To silence this warning, explicitly set the scoring parameter: scoring='neg_log_loss' for the new, scoring='accuracy' or scoring=None for the old default.
  warnings.warn(
/home/pc/.local/share/virtualenvs/zindi-9FaJJdpi/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:2150: FutureWarning: The fitted attributes of LogisticRegres

,"Cs Cs: int or list of floats, default=10Each of the values in Cs describes the inverse of regularizationstrength. If Cs is as an int, then a grid of Cs values are chosenin a logarithmic scale between 1e-4 and 1e4.Like in support vector machines, smaller values specify strongerregularization.","[0.01, 0.1, ...]"
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' might be slower in :class:`LogisticRegressionCV` because it does not handle warm-starting.- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) chosen and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations of the optimization algorithm.",1000
,"random_state random_state: int, RandomState instance, default=NoneUsed when `solver='sag'`, 'saga' or 'liblinear' to shuffle the data.Note that this only applies to the solver and not the cross-validationgenerator. See :term:`Glossary <random_state>` for details.",42
,"l1_ratios l1_ratios: array-like of shape (n_l1_ratios), default=NoneFloats between 0 and 1 passed as Elastic-Net mixing parameter (scaling betweenL1 and L2 penalties). For `l1_ratio = 0` the penalty is an L2 penalty. For`l1_ratio = 1` it is an L1 penalty. For `0 < l1_ratio < 1`, the penalty is acombination of L1 and L2.All the values of the given array-like are tested by cross-validation and theone giving the best prediction score is used... warning:: Certain values of `l1_ratios`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... deprecated:: 1.8 `l1_ratios=None` is deprecated in 1.8 and will raise an error in version 1.10. Default value will change from `None` to `(0.0,)` in version 1.10.",'warn'
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"cv cv: int or cross-validation generator, default=NoneThe default cross-validation generator used is Stratifi

In [164]:
dict_val = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(dict_val)

In [165]:
y_pred = model.predict_proba(X_val)[:,1]
accuracy = (y_val == (y_pred >= 0.5)).mean()

In [166]:
cv_scores = model.scores_[1]
cv_scores

array([[0.615     , 0.615     , 0.615     , 0.615     , 0.615     ],
       [0.61      , 0.61      , 0.61      , 0.61      , 0.61      ],
       [0.64      , 0.64      , 0.64      , 0.64      , 0.64      ],
       [0.63166667, 0.63166667, 0.63166667, 0.63166667, 0.63166667],
       [0.63333333, 0.63333333, 0.63333333, 0.63333333, 0.63333333]])

In [167]:
mean_scores = cv_scores.mean(axis=0)
mean_scores

array([0.626, 0.626, 0.626, 0.626, 0.626])

In [168]:
# Extract the scores matrix for binary classification (key 1)
cv_scores = model.scores_[1]  # Shape: (5_folds, 5_Cs)

# Compute the mean score for each C value across all folds
mean_scores = cv_scores.mean(axis=0)

# Map C values to their corresponding mean accuracy
results = pd.DataFrame({
    'C': model.Cs_,
    'Mean Accuracy': mean_scores
})

print(results)

        C  Mean Accuracy
0    0.01          0.626
1    0.10          0.626
2    1.00          0.626
3   10.00          0.626
4  100.00          0.626
